# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
MODEL = 'gpt-5-nano'
openai = OpenAI()

In [4]:
links = fetch_website_links("https://soyana.ch")
links

['https://soyana.ch/',
 'https://soyana.ch/events-news/',
 'https://soyana.ch/events-news/besuch-bei-soyana/',
 'https://soyana.ch/inserate/',
 'https://soyana.ch/events-news/',
 'https://soyana.ch/events-news/gaeste-bewirten/',
 'https://soyana.ch/soyana-lebensmittel/vegane-ernaehrung/wissenschaftliche-hintergrundinformationen-zur-ernaehrung-mit-soya/',
 'https://soyana.ch/soyana-lebensmittel/vegane-ernaehrung/vegane-ernaehrung-ihre-wissenschaftliche-begruendung/',
 'https://soyana.ch/soyana-rezepte/',
 'https://soyana.ch/sacred-star-chef-award/',
 'https://soyana.ch/events-news/newsletter/',
 'https://soyana.ch/videos/',
 'https://soyana.ch/soyana-probierpakete/',
 'https://soyana.ch/blog/',
 'https://soyana.ch/soyana-lebensmittel/',
 'https://soyana.ch/soyana-lebensmittel/was-sind-lebensmittel/',
 'https://soyana.ch/soyana-lebensmittel/nattosana/',
 'https://soyana.ch/soyana-lebensmittel/vegane-ernaehrung/',
 'https://soyana.ch/vegane-kaese-alternativen-bio/',
 'https://soyana.ch/so

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'branding page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [10]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [11]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
×
We are happy to share our intention to join forces with
NVIDIA
.
Read the announcement
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Flash-Vision-Exp
Updated
4 days ago
•
185k
•
636
Qwen/Qwen3.8-27B
Updated
22 days ago
•
6.02M
•
14k
Qwen/Qwen3.8-Flash-Next
Updated
9 days ago
•
401k
•
4.9k
zai-org/GLM-5.3-Flash
Updated
1 day ago
•
728k
•
2.06k
XHToken/Spark-X2.5-4B
Updated
2 days ago
•
4.76k
•
502
Bro

In [18]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """
brochure_system_prompt = """
  You are an assistant that analyzes the contents of several relevant pages from a company website
  and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
  Respond in markdown without code blocks.
  Include details of company culture, customers and careers/jobs if you have the information.
  """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [19]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

In [20]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [15]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the premier AI community **building the future** of machine learning. Serving as a collaboration platform, it empowers the global ML community to create, discover, and collaborate on models, datasets, and applications at scale. With over **2 million models**, **500,000+ datasets**, and more than **1 million applications**, Hugging Face is the home of machine learning innovation.

Recently, Hugging Face announced its intention to join forces with NVIDIA, further emphasizing its commitment to driving cutting-edge progress in AI.

---

## Platform & Solutions

- **Models:** Explore and use an extensive library of over 2 million pre-trained models covering NLP, vision, and more.
- **Datasets:** Access 500,000+ datasets continuously updated for machine learning tasks.
- **Spaces:** Host and discover ML-powered applications and demos — from video generation to image editing.
- **Buckets:** Scalable storage solutions integrated with Hugging Face for quick data management.
- **Enterprise Solutions:** Tailored support, inference endpoints, and enterprise-grade tools through Hugging Face PRO and dedicated team services.
- **HuggingChat:** State-of-the-art conversational AI experiences.

---

## Community & Culture

Hugging Face thrives on **open collaboration** — a vibrant community of researchers, developers, and enterprises come together to push AI boundaries.

- Active community channels on Discord, GitHub, and online forums.
- A wealth of shared resources including daily AI research papers, blog posts, and tutorials.
- Emphasis on openness, inclusivity, and knowledge sharing.
  
---

## Customers & Partners

Hugging Face serves a diverse range of customers including startups, global enterprises, academic institutions, and AI researchers. Key highlights:

- Partnership with NVIDIA to accelerate advancement in AI technologies.
- Widely adopted in industries such as tech, healthcare, finance, and media.
- Trusted for both open-source projects and enterprise deployments.

---

## Careers at Hugging Face

Join a forward-thinking team that values creativity, technical excellence, and community impact.

- Positions in machine learning research, engineering, product, and community engagement.
- Opportunity to work on state-of-the-art NLP, computer vision, and multimodal AI technologies.
- Collaborative, inclusive work environment passionate about the future of AI.

Explore current openings and join the mission to democratize machine learning.

---

## Contact & Explore

- Website: [huggingface.co](https://huggingface.co)
- Community: Discord, GitHub, Forum  
- Enterprise Inquiries: Available via website for custom solutions

---

**Hugging Face — The AI community building the future of machine learning.**  
Create, discover, collaborate — better AI starts here.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [21]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [17]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the AI community building the future of machine learning. As a collaborative platform, it empowers researchers, developers, and organizations worldwide to create, discover, and share cutting-edge machine learning models, datasets, and applications. The platform hosts over 2 million models and 500,000 datasets, fostering an open environment that accelerates innovation across AI domains like NLP, computer vision, and reinforcement learning.

Recently, Hugging Face announced an exciting partnership to join forces with **NVIDIA**, amplifying its mission to advance AI technology and deliver enterprise-grade solutions.

---

## What We Offer

- **Models**: Access and collaborate on over 2 million pre-trained and fine-tuned AI models suited for diverse tasks.
- **Datasets**: Explore hundreds of thousands of datasets spanning text, images, audio, and more—supporting groundbreaking research and application development.
- **Spaces**: Share and run interactive machine learning apps directly on the platform.
- **Enterprise Solutions**: Tailored services including Hugging Face PRO, enterprise support, inference providers, inference endpoints, and storage buckets for scalable, secure AI deployment.
- **Community Resources**: Active forums, Discord channels, GitHub repos, blogs, and educational content to connect AI practitioners globally.

---

## Our Community & Culture

Hugging Face prides itself on a vibrant, inclusive culture centered around:

- **Open Collaboration:** The platform allows unlimited hosting of public models and datasets to encourage transparent scientific progress.
- **Innovation:** Constantly surfacing trending models, datasets, and AI applications that push the boundaries of machine learning.
- **Ethics & Responsibility:** Engagement in discussions and research around AI ethics to ensure safe, equitable AI technologies.
- **Global AI Ecosystem:** Bringing together researchers, developers, enterprises, and hobbyists worldwide to build tools that advance AI for all.

---

## Who Uses Hugging Face?

- **Researchers and Academics:** For access to state-of-the-art resources and community benchmarks.
- **Developers and Startups:** To deploy AI applications rapidly with easy-to-use APIs and hosting capabilities.
- **Enterprises:** Customizable enterprise-grade AI solutions with dedicated support and infrastructure.
- **AI Enthusiasts:** A welcoming ecosystem to learn, experiment, and contribute to open source AI projects.

---

## Career Opportunities

Hugging Face is continuously growing and looking for passionate individuals who want to shape the future of AI. Opportunities span areas such as:

- Machine Learning Research and Engineering
- Software Development
- DevOps and Infrastructure
- Community Management and Developer Advocacy
- Product Management and Marketing

Join a team that values creativity, diversity, and a community-first mindset. Work alongside leading experts while contributing to projects used by millions worldwide.

---

## Connect With Us

- Visit our website to explore models, datasets, and apps: [huggingface.co](https://huggingface.co)
- Join our community on Discord and engage with AI developers globally.
- Follow our blog for research insights, case studies, and AI ethics discussions.
- Check GitHub for open source projects and collaboration opportunities.

---

Together, Hugging Face is building the future of AI — Join us on this exciting journey!

In [22]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: Where AI Meets Hugs, High-Fives, and Hyperdrive 🚀🤗

Welcome to **Hugging Face** — no, it’s not a new yoga pose but *the* AI community building the future. If machine learning were a party, we’d be the place where everyone’s sharing models, datasets, and sometimes... virtual hugs!

---

## Who Are We?

We are the **collaboration platform for the machine learning community** — hosting over **2 million models** and **500k+ datasets**. From the latest vision transformers to snazzy natural language processing models, we are the largest playground for AI explorers — researchers, developers, hobbyists (and robots, probably).

Oh, and our latest social move? Teaming up with **NVIDIA** to power up our AI universe with some serious GPU muscle. Think of it as the ultimate crossover episode of your favorite sci-fi show!

---

## What Makes Hugging Face Hug-tastic?

- **Models Galore:** Browse and deploy state-of-the-art models like Qwen, DeepSeek, GLM, and more — updated fresher than your morning latte.
- **Datasets That Don’t Ghost You:** From movie reviews (IMDB) to intricate question answering (SQuAD), we keep your AI diet balanced.
- **Spaces:** Share and explore AI apps made by the community. Want to generate a video from a picture with a simple text prompt? We got you (moving pixels, anyone?).
- **Open & Unlimited:** Host unlimited public models, datasets, and apps — because AI should be free-range and community-fed.
- **Enterprise Support:** Got AI big dreams? Our PRO & Enterprise solutions will ensure your models run faster than your morning caffeine kick.
- **HuggingChat:** Because sometimes your AI just wants to chat about the singularity, tacos, or the meaning of life.

---

## Our Culture: Fun, Friendly, and Fearlessly Collaborative 😎✨

Forget the stereotypical tech fortress. At Hugging Face, we’re:

- **Open Source Evangelists:** Sharing is caring, whether it’s code, models, or memes.
- **Community First:** We thrive on contributions from over thousands worldwide on Discord, forums, and GitHub — where your next pull request might just make AI history.
- **Innovators & Explorers:** We love a good challenge — from daily papers decoding to pushing hardware limits.
- **Doggo Approved:** Our mascot is literally a smiling, hugging yellow blob — warm, welcoming, and slightly quirky. Because AI can use some personality.

---

## Who Joins the Hug Party?

- AI Researchers wanting a collaborative lab without walls.
- Developers building the next-gen AI app that nobody knew they needed.
- Enterprises aiming to leverage robust, scalable AI with pros by their side.
- Students and AI newbies looking to learn, experiment, and eventually lead.

---

## Careers: The Place to Code, Collaborate & Make AI Magic ✨

Join us if you want to:

- Work with cutting-edge AI alongside some of the coolest nerds on the planet.
- Help build tools used by *millions* every day.
- Grow in a community that celebrates innovation, diversity, and a good sense of humor.
- Enjoy working somewhere that’s as passionate about open source as it is about a good joke or two.

---

## Ready to Hug the Future?

Jump into the AI revolution with **Hugging Face: The Home of Machine Learning**. Whether you're looking to build, browse, or just blend in with a warm, buzzing community of AI enthusiasts — we’re here, with arms wide open (and CPUs humming).

**Explore →** Browse 2M+ models | Dive into Datasets | Join Spaces | Say hi on [Discord]

---

### Official Colors of Hug:

- Sunbeam Yellow: #FFD21E
- Orange Zest: #FF9D00
- Smokin’ Gray: #6B7280

---

_Hugging Face: We put the "fun" in "function approximation."_

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>